In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict


# =========================================================
# 1. STATE
# =========================================================

class State(TypedDict):
    count: int


# =========================================================
# 2. NODES
# =========================================================

def add_one(state: State):

    print("ADD_ONE running...")

    return {
        "count": state["count"] + 1
    }


def add_two(state: State):

    print("ADD_TWO running...")

    return {
        "count": state["count"] + 2
    }


def add_three(state: State):

    print("ADD_THREE running...")

    return {
        "count": state["count"] + 3
    }


# =========================================================
# 3. GRAPH
# =========================================================

builder = StateGraph(State)

builder.add_node("add_one", add_one)
builder.add_node("add_two", add_two)
builder.add_node("add_three", add_three)

builder.add_edge(START, "add_one")
builder.add_edge("add_one", "add_two")
builder.add_edge("add_two", "add_three")
builder.add_edge("add_three", END)


# =========================================================
# 4. CHECKPOINTER
# =========================================================

checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)


# =========================================================
# 5. ORIGINAL THREAD
# =========================================================

config = {
    "configurable": {
        "thread_id": "thread-001"
    }
}


print("\n================================")
print("ORIGINAL EXECUTION")
print("================================")

original_result = graph.invoke(
    {"count": 0},
    config=config
)

print(
    "Original Result:",
    original_result
)


# =========================================================
# 6. GET HISTORY
# =========================================================

history = list(
    graph.get_state_history(config)
)


print("\n================================")
print("CHECKPOINT HISTORY")
print("================================")

for index, state in enumerate(history):

    print(
        f"\nCheckpoint {index}"
    )

    print(
        "ID:",
        state.config[
            "configurable"
        ]["checkpoint_id"]
    )

    print(
        "Values:",
        state.values
    )

    print(
        "Next:",
        state.next
    )


# =========================================================
# 7. FIND HISTORICAL CHECKPOINT
# =========================================================

# History normally newest -> oldest.
# We want a checkpoint where execution can continue.

target_state = None

for state in history:

    if state.next:

        target_state = state
        break


if target_state is None:

    print(
        "\nNo checkpoint available for replay."
    )

else:

    checkpoint_id = (
        target_state.config[
            "configurable"
        ]["checkpoint_id"]
    )

    historical_config = {
        "configurable": {
            "thread_id": "thread-001",
            "checkpoint_id": checkpoint_id
        }
    }


    # =====================================================
    # 8. READ HISTORICAL STATE
    # =====================================================

    historical_state = graph.get_state(
        historical_config
    )

    print("\n================================")
    print("HISTORICAL CHECKPOINT")
    print("================================")

    print(
        "Checkpoint ID:",
        checkpoint_id
    )

    print(
        "Values:",
        historical_state.values
    )

    print(
        "Next:",
        historical_state.next
    )


    # =====================================================
    # 9. REPLAY / CONTINUE
    # =====================================================

    print("\n================================")
    print("REPLAY FROM CHECKPOINT")
    print("================================")

    replay_result = graph.invoke(
        None,
        config=historical_config
    )

    print(
        "\nReplay Result:",
        replay_result
    )


    # =====================================================
    # 10. FINAL STATE AFTER REPLAY
    # =====================================================

    replay_state = graph.get_state(
        historical_config
    )

    print("\n================================")
    print("REPLAY FINAL STATE")
    print("================================")

    print(
        "Values:",
        replay_state.values
    )

    print(
        "Next:",
        replay_state.next
    )


ORIGINAL EXECUTION
ADD_ONE running...
ADD_TWO running...
ADD_THREE running...
Original Result: {'count': 6}

CHECKPOINT HISTORY

Checkpoint 0
ID: 1f1bcfa9-a97e-6257-8003-73ff29e0a232
Values: {'count': 6}
Next: ()

Checkpoint 1
ID: 1f1bcfa9-a975-6e56-8002-70e7f2a9b293
Values: {'count': 3}
Next: ('add_three',)

Checkpoint 2
ID: 1f1bcfa9-a930-6dfa-8001-80186c33f261
Values: {'count': 1}
Next: ('add_two',)

Checkpoint 3
ID: 1f1bcfa9-a8c1-6e0a-8000-6a9cf3893df6
Values: {'count': 0}
Next: ('add_one',)

Checkpoint 4
ID: 1f1bcfa9-a872-68e8-bfff-4be38b83f5e2
Values: {}
Next: ('__start__',)

HISTORICAL CHECKPOINT
Checkpoint ID: 1f1bcfa9-a975-6e56-8002-70e7f2a9b293
Values: {'count': 3}
Next: ('add_three',)

REPLAY FROM CHECKPOINT
ADD_THREE running...

Replay Result: {'count': 6}

REPLAY FINAL STATE
Values: {'count': 3}
Next: ('add_three',)
